<h2 style="color:#16A085; margin-bottom:6px;">
  03 – MLP Classifier (From-Scratch Neural Network)
</h2>

<p style="font-size:16px; margin-top:4px;">
  <strong>Overview:</strong>
  This notebook implements a from-scratch neural network model for
  multi-label emotion classification using a
  <strong>Multi-Layer Perceptron (MLP)</strong>.
</p>

<p style="font-size:16px;">
  <strong>Goal:</strong>
  To evaluate whether a simple neural architecture can capture
  non-linear patterns in TF-IDF feature representations and improve
  performance over traditional linear models.
</p>

<p style="font-size:16px;">
  The model is built using <code>MLPClassifier</code> with a custom
  configuration and wrapped in a One-vs-Rest strategy to support
  multi-label prediction. This experiment serves as a transition from
  traditional machine learning approaches to deeper neural and
  pretrained transformer-based models.
</p>


In [2]:
#wandb login check to make sure everything goes smooth

from kaggle_secrets import UserSecretsClient
import wandb

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=api_key)

print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sharmilam-official (sharmila-m) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful!


In [3]:
# 03 – MLP Classifier (From-Scratch Neural Network on TF-IDF Features)

import pandas as pd
import numpy as np
import wandb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.multiclass import OneVsRestClassifier

print("Started")

label_cols = ["anger", "fear", "joy", "sadness", "surprise"]

# -------------------------
# 1. W&B Init
# -------------------------
wandb.init(
    project="24ds2000048-t32025",
    entity="sharmila-m",
    name="03_mlp_classifier_from_scratch",
    config={
        "max_features": 20000,
        "ngram_range": (1, 2),
        "hidden_units": 256,
        "learning_rate": 1e-3,
        "max_iter": 20,
        "threshold": 0.5,
        "test_size": 0.2,
        "random_state": 42
    }
)
config = wandb.config

# -------------------------
# 2. Load data
# -------------------------
print("Loading the data...")
df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/train.csv")

train_df, val_df = train_test_split(
    df,
    test_size=config.test_size,
    random_state=config.random_state
)

# -------------------------
# 3. TF-IDF
# -------------------------
tfv = TfidfVectorizer(
    max_features=config.max_features,
    ngram_range=tuple(wandb.config.ngram_range),
)

tfv.fit(train_df["text"])

X_train = tfv.transform(train_df["text"])
X_val   = tfv.transform(val_df["text"])

y_train = train_df[label_cols].values
y_val   = val_df[label_cols].values

# -------------------------
# 4. MLP model (One-vs-Rest)
# -------------------------
model = OneVsRestClassifier(
    MLPClassifier(
        hidden_layer_sizes=(config.hidden_units,),
        activation="relu",
        solver="adam",
        learning_rate_init=config.learning_rate,
        max_iter=config.max_iter,
        verbose=False,
        random_state=config.random_state
    )
)

# -------------------------
# 5. Train
# -------------------------
print("Training the MLP model...")
model.fit(X_train, y_train)

# -------------------------
# 6. Validation F1 + Accuracy
# -------------------------
preds_proba = model.predict_proba(X_val)
preds_bin = (preds_proba > config.threshold).astype(int)

val_f1_macro = f1_score(y_val, preds_bin, average="macro")
val_accuracy = accuracy_score(y_val, preds_bin)

print("MLP Validation Macro F1:", val_f1_macro)
print("MLP Validation Accuracy:", val_accuracy)

wandb.log({
    "val_f1_macro": val_f1_macro,
    "val_accuracy": val_accuracy
})

wandb.finish()

Started


Loading the data...
Training the MLP model...


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Validation Macro F1: 0.7296872717265669
MLP Validation Accuracy: 0.5849194729136163


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


val_accuracy,▁
val_f1_macro,▁
val_accuracy,0.58492
val_f1_macro,0.72969


In [4]:
# -------------------------
# 7. Test Predictions
# -------------------------
print("Generating test predictions...")
test_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")
X_test = tfv.transform(test_df["text"].fillna(""))

test_preds_proba = model.predict_proba(X_test)
test_preds_bin = (test_preds_proba > config.threshold).astype(int)

submission = pd.DataFrame(test_preds_bin, columns=label_cols)

if "id" in test_df.columns:
    submission.insert(0, "id", test_df["id"])

submission_filename = "submission.csv"
submission.to_csv(submission_filename, index=False)
print(f"Saved {submission_filename}")

Generating test predictions...
Saved submission.csv


<hr/>

<h3 style="color:#16A085;">
  Results & Runtime Summary
</h3>

<p style="font-size:16px;">
  <strong>Model:</strong> TF-IDF + MLP Classifier (One-vs-Rest)
</p>

<p style="font-size:16px;">
  <strong>Validation Metrics (Internal):</strong><br/>
  Macro F1 Score: <strong>0.7297</strong><br/>
  Accuracy: <strong>0.585</strong>
</p>

<p style="font-size:16px;">
  <strong>Kaggle Public Leaderboard Score:</strong><br/>
  Macro F1 Score: <strong>0.713</strong>
</p>

<p style="font-size:16px;">
  <strong>Runtime Environment:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li><strong>Platform:</strong> Kaggle Notebook</li>
  <li><strong>Hardware:</strong> CPU</li>
  <li><strong>Estimated Runtime:</strong> ~7–8 minutes</li>
  <li><strong>Libraries:</strong> scikit-learn, pandas, NumPy</li>
</ul>

<p style="font-size:16px;">
  <strong>Observations:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li>
    The MLP model outperforms both the baseline and tuned logistic
    regression models in terms of Macro F1 score.
  </li>
  <li>
    The improvement indicates that even a shallow neural network can
    capture useful non-linear relationships in TF-IDF features.
  </li>
  <li>
    While accuracy remains moderate due to the multi-label nature of
    the task, the Macro F1 score confirms balanced performance across
    all emotion categories.
  </li>
</ul>
